In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

DATA_DIR = BASE_DIR / "data"
REY_DIR = DATA_DIR / "REY_DATASET"
REY_ROI_ROT0_DIR = REY_DIR / "REY_roi_rot0"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

CSV_LIMPIO = TABLES_DIR / "rey_traza_limpia.csv"

print("TensorFlow:", tf.__version__)
print("CSV limpio existe:", CSV_LIMPIO.exists())
print("REY_ROI_ROT0_DIR existe:", REY_ROI_ROT0_DIR.exists())

In [ ]:
df = pd.read_csv(CSV_LIMPIO)

print("Número de muestras:", len(df))
display(df.head())

print("\nDistribución de rotaciones:")
print(df["rot"].value_counts().sort_index())

In [ ]:
rot_to_class = {
    -90: 0,
    0: 1,
    90: 2,
    180: 3
}

class_to_rot = {
    0: -90,
    1: 0,
    2: 90,
    3: 180
}

df["clase"] = df["rot"].map(rot_to_class)

print(df[["fname", "rot", "clase"]].head())

print("\nDistribución por clase:")
print(df["clase"].value_counts().sort_index())

In [ ]:
IMG_SIZE = 160
N_CLASSES = 4
RANDOM_STATE = 7

print("Tamaño de entrada:", IMG_SIZE, "x", IMG_SIZE)
print("Número de clases:", N_CLASSES)

In [ ]:
def cargar_roi_y_clase(fila):
    """
    Lee una ROI de REY_roi_rot0 y devuelve:
    - imagen redimensionada y normalizada
    - clase de rotación
    """
    
    img_path = REY_ROI_ROT0_DIR / fila["fname"]
    
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    
    if img is None:
        raise ValueError("No se pudo leer la ROI: " + str(img_path))
    
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=-1)
    
    clase = int(fila["clase"])
    
    return img, clase

In [ ]:
X = []
y = []
nombres = []

for i, fila in df.iterrows():
    img, clase = cargar_roi_y_clase(fila)
    
    X.append(img)
    y.append(clase)
    nombres.append(fila["fname"])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)
nombres = np.array(nombres)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Primeras clases:", y[:10])

In [ ]:
X_train, X_temp, y_train, y_temp, names_train, names_temp = train_test_split(
    X, y, nombres,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test, names_val, names_test = train_test_split(
    X_temp, y_temp, names_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nDistribución train:", np.bincount(y_train))
print("Distribución val:", np.bincount(y_val))
print("Distribución test:", np.bincount(y_test))

In [ ]:
clases_unicas = np.unique(y_train)

pesos = compute_class_weight(
    class_weight="balanced",
    classes=clases_unicas,
    y=y_train
)

class_weight = {}

for clase, peso in zip(clases_unicas, pesos):
    class_weight[int(clase)] = float(peso)

print("Pesos de clase:")
print(class_weight)

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    
    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    
    layers.Dense(N_CLASSES, activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
checkpoint_path = MODELS_DIR / "rey_rotation_keras_best.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=8,
        restore_best_weights=True,
        mode="max",
        verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=16,
    callbacks=callbacks,
    class_weight=class_weight
)

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()

fig_path = FIGURES_DIR / "rey_rotation_keras_training_curves.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)

print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

In [ ]:
probs = model.predict(X_test)
y_pred = np.argmax(probs, axis=1)

cm = confusion_matrix(y_test, y_pred)

print("Matriz de confusión:")
print(cm)

print("\nReporte de clasificación:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["-90", "0", "90", "180"]
))

In [ ]:
labels_rot = ["-90", "0", "90", "180"]

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Matriz de confusión - Rotación Keras")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.xticks(np.arange(4), labels_rot)
plt.yticks(np.arange(4), labels_rot)

for i in range(4):
    for j in range(4):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()

fig_path = FIGURES_DIR / "rey_rotation_keras_confusion_matrix.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)

In [ ]:
resultados_rotacion = pd.DataFrame({
    "imagen": names_test,
    "clase_real": y_test,
    "rot_real": [class_to_rot[int(c)] for c in y_test],
    "clase_pred": y_pred,
    "rot_pred": [class_to_rot[int(c)] for c in y_pred],
    "prob_max": np.max(probs, axis=1)
})

csv_resultados = TABLES_DIR / "rey_rotation_keras_resultados_test.csv"
resultados_rotacion.to_csv(csv_resultados, index=False)

print("Resultados guardados en:", csv_resultados)
display(resultados_rotacion.head())

In [ ]:
n_ver = 8
indices = np.arange(len(X_test))

plt.figure(figsize=(12, 8))

for i in range(min(n_ver, len(indices))):
    idx = indices[i]
    
    img = X_test[idx].squeeze()
    
    rot_real = class_to_rot[int(y_test[idx])]
    rot_pred = class_to_rot[int(y_pred[idx])]
    
    plt.subplot(2, 4, i + 1)
    plt.imshow(img, cmap="gray")
    plt.title(f"real={rot_real}, pred={rot_pred}")
    plt.axis("off")

plt.tight_layout()

fig_path = FIGURES_DIR / "rey_rotation_keras_examples.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)

In [ ]:
tabla_rotation_keras = pd.DataFrame({
    "modelo": ["CNN Keras"],
    "accuracy_test": [test_acc],
    "loss_test": [test_loss],
    "macro_f1_aprox": [0.84],
    "weighted_f1_aprox": [0.92],
    "comentario": ["Buen resultado global; la clase 180 tiene pocos ejemplos"]
})

tabla_path = TABLES_DIR / "rey_rotation_keras_resumen_test.csv"
tabla_rotation_keras.to_csv(tabla_path, index=False)

display(tabla_rotation_keras)
print("Tabla guardada en:", tabla_path)